In [0]:
# =============================================================================
# ICEBASE — PHASE 2 | NOTEBOOK 04
# Silver fact_tickets — Streaming Table (Python)
# Idaho Mashers Hockey Analytics Platform
# =============================================================================
# PIPELINE SOURCE FILE — DO NOT RUN AS A STANDALONE NOTEBOOK.
# Add to the icebase-bronze-to-silver pipeline as a source file.
#
# LANGUAGE: Python (required — SQL cannot union two independent readStream sources)
#
# TABLE DEFINED HERE:
#   icebase.silver.fact_tickets  (Streaming Table)
#
# SOURCES (two streams unioned):
#   1. icebase.bronze.raw_tickets        — Phase 1 seed history (~47k tickets)
#   2. icebase.bronze.raw_tickets_stream — Simulator volume drops (ongoing)
#
# ENRICHMENT ADDED IN SILVER:
#   seat_tier_rank     : Numeric 1–5 rank by section (rinkside=5 is premium)
#                        Enables price tier analysis and ML feature engineering
#   days_before_game   : Days between purchase_ts and game_date
#                        Negative = day-of walk-up, positive = advance purchase
#   is_advance_purchase: TRUE if bought more than 3 days before game
#                        Key behavioral signal for fan loyalty scoring
#
# QUALITY GATES:
#   expect_or_drop : hard DROP on null IDs and zero/negative prices
#                    Bad records are counted in the DQ report but not kept
#   expect         : soft WARN on section_tier and channel values
#                    Unexpected values flagged but record preserved
#
# NOTE ON QUARANTINE:
#   Records dropped here by expect_or_drop do NOT automatically go anywhere.
#   Notebook 06 (quarantine_tickets) reads the same raw source and
#   explicitly filters to bad records — capturing them for investigation.
# =============================================================================

from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# -----------------------------------------------------------------------------
# Section tier rank mapping
# Higher rank = more premium seat = higher expected price
# Used as an ordinal feature in both revenue analysis and ML models
# -----------------------------------------------------------------------------

TIER_RANK = {
    "rinkside":   5,
    "lower_bowl": 4,
    "mid_bowl":   3,
    "upper_bowl": 2,
    "standing":   1,
}

# -----------------------------------------------------------------------------
# Helper: enrich a raw ticket DataFrame with Silver-level derived columns.
# Applied identically to both seed and simulator sources.
# -----------------------------------------------------------------------------

def _enrich_tickets(df):
    """Apply Silver enrichment to a raw ticket DataFrame."""

    # Build a Spark map literal from the Python dict
    tier_map = F.create_map(
        *[item for pair in [
            (F.lit(k), F.lit(v)) for k, v in TIER_RANK.items()
        ] for item in pair]
    )

    return df.select(
        F.col("ticket_id"),
        F.col("customer_id"),
        F.col("game_id"),
        F.col("game_date").cast(DateType()).alias("game_date"),
        F.col("section_tier"),

        # Numeric seat tier rank — rinkside=5 is most premium
        tier_map[F.col("section_tier")].alias("seat_tier_rank"),

        F.col("seat_row"),
        F.col("seat_number"),
        F.col("ticket_price"),
        F.col("purchase_channel"),
        F.col("purchase_ts"),
        F.col("is_promo_ticket"),
        F.col("is_jersey_night_game"),
        F.col("is_lapsed_reactivation"),
        F.col("season_phase"),

        # Days before game purchased
        # Positive = bought in advance (planned fan)
        # Zero/negative = day-of walk-up (impulse buyer)
        F.datediff(
            F.col("game_date").cast(DateType()),
            F.to_date(F.col("purchase_ts"))
        ).alias("days_before_game"),

        # Advance purchase flag: bought more than 3 days out
        # Key signal for fan loyalty and season holder likelihood
        F.when(
            F.datediff(
                F.col("game_date").cast(DateType()),
                F.to_date(F.col("purchase_ts"))
            ) > 3,
            True
        ).otherwise(False).alias("is_advance_purchase"),

        F.col("_ingested_at"),
    )


# -----------------------------------------------------------------------------
# fact_tickets — Streaming Table
# -----------------------------------------------------------------------------

@dp.table(
    comment          = "Silver ticket fact — enriched transactions from seed + simulator",
    table_properties = {"quality": "silver", "team": "idaho_mashers"}
)
@dp.expect_or_drop("ticket_id_not_null",   "ticket_id IS NOT NULL")
@dp.expect_or_drop("customer_id_not_null", "customer_id IS NOT NULL")
@dp.expect_or_drop("game_id_not_null",     "game_id IS NOT NULL")
@dp.expect_or_drop("price_positive",       "ticket_price > 0")
@dp.expect(
    "valid_section_tier",
    "section_tier IN ('rinkside','lower_bowl','mid_bowl','upper_bowl','standing')"
)
@dp.expect(
    "valid_purchase_channel",
    "purchase_channel IN ('online','box_office','mobile_app','reseller')"
)
def fact_tickets():

    # ── Source 1: Phase 1 seed ticket history ────────────────────────────
    seed_tickets = _enrich_tickets(
        spark.readStream
            .option("skipChangeCommits", "true")
            .table("icebase.bronze.raw_tickets")
    )

    # ── Source 2: Simulator volume drops ─────────────────────────────────
    # Fully qualified name — pipeline default schema is silver
    new_tickets = _enrich_tickets(
        spark.readStream
            .option("skipChangeCommits", "true")
            .table("icebase.bronze.raw_tickets_stream")
    )

    return seed_tickets.union(new_tickets)